# 01 · The model

Every agent starts with a model call. In this notebook you connect CrewAI's `LLM` to the shared local
model, make a first call, and get a structured answer back: ShipFast's triage, which sorts a customer
message into an intent and an order ID.

**You will learn**

- how the shared model is set up in CrewAI, and why its settings are fixed
- how to call CrewAI's `LLM` directly, with text and with messages
- how to get a typed, validated answer instead of free text
- where the token counts of your calls are kept

**Before you start:** Ollama is running (`ollama serve`) and the shared model exists
(`ollama create qwen3-agents -f contracts/ollama/Modelfile`). See the README.

In [1]:
# Import crew_agents before anything else from CrewAI: it switches CrewAI's telemetry off.
from importlib.metadata import version

from crew_agents.llm import make_llm

from agent_contracts.harness import load_settings, verify_model

# Check that Ollama is running and the shared model exists. If not, this stops with the command that fixes it.
verify_model()

print("Model: ", load_settings().model)
print("CrewAI:", version("crewai"))

Model:  qwen3-agents
CrewAI: 1.15.34


## The shared model

All three tutorials (LangGraph, CrewAI and Strands) use the same model with the same settings, so any
difference you see later comes from the framework, not the model:

| Setting | Value | Why |
|---|---|---|
| model | `qwen3-agents` | `qwen3:8b`, a small model that runs on a laptop, with the settings below built in |
| `temperature` | 0 | the most likely answer every time |
| `seed` | 42 | makes repeated runs repeatable |
| `num_ctx` | 16384 | the model's working memory in tokens; Ollama's default of 4096 is too small for agents |
| thinking | off | Qwen 3 can reason before answering; off is 8 to 10 times faster on this machine |

CrewAI 1.15 talks to Ollama through its **OpenAI-compatible API**, not Ollama's own. That has two
consequences you can see in `make_llm()`: thinking is turned off the OpenAI way, with
`reasoning_effort="none"`, and the context size cannot be sent per request at all. That second point is
why the shared settings are written into the `qwen3-agents` model itself (`contracts/ollama/Modelfile`):
it is the only way CrewAI gets the same 16K context as the other two frameworks.

In [2]:
# make_llm() lives in crew_agents/llm.py, so these notebooks and the finished agents build the model the
# same way. Print its source to see every setting it passes to CrewAI's LLM.
import inspect

print(inspect.getsource(make_llm))

llm = make_llm()  # the model used in the rest of this notebook
print("CrewAI chose this provider class:", type(llm).__name__)

def make_llm(settings: ModelSettings | None = None) -> LLM:
    s = settings or load_settings()
    return LLM(
        model=f"ollama/{s.model}",
        base_url=s.host,
        temperature=s.temperature,
        seed=s.seed,
        reasoning_effort="none",  # thinking off on the OpenAI-style API
        context_window_size=s.num_ctx,  # otherwise CrewAI assumes 8,192 and trims early
    )

CrewAI chose this provider class: OpenAICompatibleCompletion


## A first call

`llm.call` takes a string (or a list of messages) and returns the reply as a string. Ask about something
the model cannot know: ShipFast is a made-up shop, and its policy is not in the model's training data.

In [3]:
# Ask something the model cannot know: ShipFast is a made-up shop. call() returns the reply as a string.
answer = llm.call("In one sentence: what is ShipFast's refund policy for lost packages?")

print("Model's answer:", answer)

Model's answer: ShipFast's refund policy for lost packages typically involves a full refund of the shipping fees once the package is confirmed lost, provided the customer submits proof of the lost package and follows the company's claim process.


The model either says it does not know or makes something up. Both are the point of the next
notebook: a model on its own only knows what it was trained on, and tools are how it reaches real data.

## Messages and roles

A conversation is a list of messages. CrewAI uses plain dictionaries in the OpenAI style: a `system`
message sets the rules, `user` messages are the customer, and the model's replies are `assistant`
messages.

In [4]:
messages = [
    # The system message sets the rules for the whole conversation.
    {"role": "system", "content": "You are ShipFast's support assistant. Reply in one short sentence. "
                                  "You cannot look anything up yet, so never invent order details."},
    # The user message is what the customer typed.
    {"role": "user", "content": "Hi, where is my order A1001?"},
]

print("Reply:", llm.call(messages))

Reply: Your order A1001 is currently being processed and will be shipped soon.


If that reply states anything about order A1001, such as a status or a delivery date, the model
invented it: it has no access to orders. A rule in the system message makes invention less likely, but
it is not a guarantee, especially with a small model. That is why ShipFast checks facts with tools
(notebook 02) and checks every refund against a ledger in code, never against the model's word.

## Structured output: triage

Free text is fine for a person, but code needs fields it can check. ShipFast's first step is **triage**:
read the customer's message and return an `intent` and an `order_id`. The shape is the shared `Triage`
model from `agent_contracts`, the same class the LangGraph and Strands builds use.

First, look at the shape itself: the fields the model must fill in.

In [5]:
# Triage is a Pydantic model shared by all three tutorials (contracts/agent_contracts/shipfast/schema.py).
from agent_contracts.shipfast.schema import Triage

# Its JSON schema is what the model gets to see. Print each field with its allowed values and description.
for name, field in Triage.model_json_schema()["properties"].items():
    allowed = field.get("enum") or [option["type"] for option in field["anyOf"]]
    print(name)
    print("  allowed:    ", ", ".join(allowed))
    print("  description:", field["description"])

intent
  allowed:     order_status, refund_request, complaint, other
  description: order_status, refund_request, complaint, or other (out of scope)
order_id
  allowed:     string, null
  description: The order ID, such as A1001, or null if there is none


Two fields, and `intent` can only be one of four values. Pass the class as `response_model` and `call`
returns a validated `Triage` object instead of a string. The prompt explains what each intent means.

In [6]:
# The instructions for triage: what each intent means, and when order_id should be null.
TRIAGE_PROMPT = (
    "Classify the customer's latest message for ShipFast support.\n"
    "intent: order_status (where is my order, when will it arrive), refund_request (an explicit ask for a "
    "refund, return or cancellation), complaint (something is wrong, with no explicit refund ask), or other "
    "(anything that is not about orders, deliveries, refunds or complaints).\n"
    "order_id: the order ID such as A1001 if the message gives one, else null."
)


def triage(text: str) -> Triage:
    """Triage one customer message. Returns a validated Triage object, not a string."""
    messages = [
        {"role": "system", "content": TRIAGE_PROMPT},
        {"role": "user", "content": text},
    ]
    return llm.call(messages, response_model=Triage)  # response_model: parse the reply into a Triage


result = triage("My order A1002 never arrived. I want a refund.")

print("Returned:", type(result).__name__)
print("intent:  ", result.intent)
print("order_id:", result.order_id)

Returned: Triage
intent:   refund_request
order_id: A1002


Now run it over the opening message of every ShipFast scenario and compare with the expected answer
from `scenarios.json`. (Some scenarios have a second turn that only makes sense with the first one in
memory; that is notebook 04.)

In [7]:
# Triage the first message of every ShipFast scenario and compare it with the expected answer.
import time

from agent_contracts.shipfast.scoring import load_scenarios

seen = set()  # opening messages already tried: two scenarios start with the same one
correct = 0

print(f"{'check':6} {'time':>5}  {'intent':15} {'order':6} customer message")
for scenario in load_scenarios():
    text = scenario["turns"][0]  # the customer's first message
    if text in seen:
        continue
    seen.add(text)
    expected = scenario["expected"]  # the right intent and order ID, from scenarios.json

    start = time.perf_counter()
    got = triage(text)
    seconds = time.perf_counter() - start

    ok = (got.intent, got.order_id) == (expected["intent"], expected["order_id"])
    correct += ok
    print(f"{'ok' if ok else 'MISS':6} {seconds:4.1f}s  {got.intent:15} {str(got.order_id):6} {text}")

print(f"\n{correct} of {len(seen)} messages triaged correctly")

check   time  intent          order  customer message


ok      3.8s  order_status    A1001  Hi, where is my order A1001?


ok      4.2s  refund_request  A1002  My order A1002 never arrived and tracking says it's lost. I want a refund.


ok      4.2s  refund_request  A1003  Order A1003 is more than a week late. Please cancel it and refund me.


ok      3.8s  complaint       A1004  My vase set from order A1004 arrived broken.


ok      3.7s  refund_request  A1005  I want to return order A1005, the stand wobbles.


ok      3.9s  order_status    A9999  Where is order A9999?


ok      2.8s  other           None   Can you write me a poem about cats?

7 of 7 messages triaged correctly


## What the calls cost

CrewAI's `LLM` keeps a running total of the tokens it has used across every call made with it. Notebook
07 connects CrewAI to the shared harness instead, so all three frameworks are counted the same way.

In [8]:
# CrewAI's LLM keeps a running total over every call made with it, so measure one call as a difference.
before = llm.get_token_usage_summary()
triage("Where is order A9999?")
after = llm.get_token_usage_summary()

print("This call")
print("  input tokens: ", after.prompt_tokens - before.prompt_tokens, "(instructions, schema and message)")
print("  output tokens:", after.completion_tokens - before.completion_tokens, "(the model's answer)")
print("Every call in this notebook so far")
print("  calls:        ", after.successful_requests)
print("  total tokens: ", after.total_tokens)

This call
  input tokens:  121 (instructions, schema and message)
  output tokens: 23 (the model's answer)
Every call in this notebook so far
  calls:         11
  total tokens:  1485


## How the others do this

- **LangGraph** (through LangChain) uses `ChatOllama`, which talks to Ollama's own API, so it can send
  every setting per request, `num_ctx` included, and turns thinking off with `reasoning=False`. Replies
  are message objects (`AIMessage`) rather than strings, and structured output is
  `llm.with_structured_output(Triage)`.
- **Strands** uses `OllamaModel(host, model_id="qwen3-agents", additional_args={"think": False})`, also
  on Ollama's own API.

All three get the same `Triage` class from `agent_contracts`.

## Takeaways

- CrewAI's `LLM` is the model; `call` takes text or messages and returns text, or a validated object with
  `response_model`.
- CrewAI reaches Ollama through the OpenAI-compatible API, so some settings travel in the model, not the
  request.
- Fixing the model's settings is what makes a framework comparison fair.
- A model alone cannot see ShipFast's orders or policy. Next: **02 · Tools**.